# 2 — Interface de treinamento da GRU

Este notebook treina a desagregação diária para:

| Intervalo de saída | Neurônios de saída |
|---|---:|
| 1 hora | 24 |
| 2 horas | 12 |
| 4 horas | 6 |

As janelas possíveis são 3, 5 e 7 dias. A saída usa `Softmax`, portanto as frações somam 1 e a precipitação prevista sempre conserva o total do dia central.

O conjunto é dividido cronologicamente em treino (70%), validação (15%) e teste (15%). Janelas que atravessam as fronteiras são eliminadas. Por padrão, apenas dias centrais chuvosos e completos são usados para não inflar artificialmente as métricas com dias secos.

In [1]:
# Se faltar alguma biblioteca, descomente e execute uma vez:
# %pip install torch pandas numpy scikit-learn matplotlib seaborn ipywidgets

from pathlib import Path
import copy
import json
import math
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError as e:
    raise ImportError("PyTorch não está instalado. Execute a linha de instalação acima.") from e

from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display, clear_output

sns.set_theme(style="whitegrid")

## Configuração sem interface gráfica

In [2]:
CAMINHO_DADOS = Path("dados_processados/chuva_horaria_padronizada.csv")
PASTA_MODELOS = Path("modelos")
PASTA_RESULTADOS = Path("resultados")

# Estas variáveis também podem ser escolhidas pelos controles no final.
INTERVALO_HORAS = 1      # 1, 2 ou 4 -> 24, 12 ou 6 saídas
JANELA_DIAS = 5          # 3, 5 ou 7
NEURONIOS_OCULTOS = 32   # tamanho do estado oculto
NUM_CAMADAS = 1
EPOCAS = 200
BATCH_SIZE = 64
TAXA_APRENDIZAGEM = 1e-3
PACIENCIA = 20
LIMIAR_CHUVA_MM = 0.1
REPETICAO = 1
SEMENTE_BASE = 42

assert INTERVALO_HORAS in {1, 2, 4}
assert JANELA_DIAS in {3, 5, 7}
assert JANELA_DIAS % 2 == 1

## Preparação das amostras

In [3]:
def fixar_semente(semente):
    random.seed(semente)
    np.random.seed(semente)
    torch.manual_seed(semente)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semente)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def carregar_horario(caminho):
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(
            f"Não encontrei {caminho.resolve()}. Execute primeiro o notebook 01."
        )
    df = pd.read_csv(caminho, parse_dates=["data_hora"])
    exigidas = {"data_hora", "precipitacao_mm"}
    if not exigidas.issubset(df.columns):
        raise ValueError(f"O arquivo deve conter {sorted(exigidas)}")
    df = df.sort_values("data_hora").set_index("data_hora")
    df["precipitacao_mm"] = pd.to_numeric(df["precipitacao_mm"], errors="coerce")
    return df


def construir_amostras(horario, janela_dias, intervalo_horas, limiar_chuva=0.1):
    if 24 % intervalo_horas != 0:
        raise ValueError("O intervalo deve dividir 24 horas exatamente.")
    metade = janela_dias // 2
    grupos = horario["precipitacao_mm"].groupby(horario.index.normalize())
    arrays = {}
    for data, serie in grupos:
        serie = serie.sort_index()
        if len(serie) == 24 and serie.notna().all():
            arrays[pd.Timestamp(data)] = serie.to_numpy(dtype=np.float32)

    datas_validas = sorted(arrays)
    if len(datas_validas) < 100:
        raise ValueError(f"Há somente {len(datas_validas)} dias completos. Verifique os dados.")
    totais = {d: float(arrays[d].sum()) for d in datas_validas}
    conjunto_datas = set(datas_validas)

    X, Y, datas, inicios, fins = [], [], [], [], []
    for central in datas_validas:
        janela = [central + pd.Timedelta(days=i) for i in range(-metade, metade + 1)]
        if not all(d in conjunto_datas for d in janela):
            continue
        if totais[central] < limiar_chuva:
            continue
        x = np.array([totais[d] for d in janela], dtype=np.float32)[:, None]
        y_h = arrays[central]
        y = y_h.reshape(-1, intervalo_horas).sum(axis=1).astype(np.float32)
        if not np.isclose(y.sum(), totais[central], atol=1e-4):
            raise RuntimeError(f"Falha de conservação na preparação de {central.date()}")
        X.append(x); Y.append(y); datas.append(central)
        inicios.append(janela[0]); fins.append(janela[-1])

    if not X:
        raise ValueError("Nenhuma amostra chuvosa pôde ser construída.")
    return {
        "X": np.stack(X), "Y": np.stack(Y),
        "datas": np.array(datas, dtype="datetime64[ns]"),
        "inicios": np.array(inicios, dtype="datetime64[ns]"),
        "fins": np.array(fins, dtype="datetime64[ns]"),
        "datas_validas": np.array(datas_validas, dtype="datetime64[ns]"),
    }


def dividir_cronologicamente(amostras, proporcao_treino=0.70, proporcao_val=0.15):
    datas_base = amostras["datas_validas"]
    n = len(datas_base)
    i_treino = max(0, min(n - 2, int(n * proporcao_treino) - 1))
    i_val = max(i_treino + 1, min(n - 1, int(n * (proporcao_treino + proporcao_val)) - 1))
    corte_treino = datas_base[i_treino]
    corte_val = datas_base[i_val]

    mascaras = {
        "treino": amostras["fins"] <= corte_treino,
        "validacao": (amostras["inicios"] > corte_treino) & (amostras["fins"] <= corte_val),
        "teste": amostras["inicios"] > corte_val,
    }
    partes = {}
    for nome, mascara in mascaras.items():
        partes[nome] = {k: v[mascara] for k, v in amostras.items() if k != "datas_validas"}
        if len(partes[nome]["X"]) == 0:
            raise ValueError(f"O conjunto {nome} ficou vazio.")
    cortes = {
        "fim_treino": str(pd.Timestamp(corte_treino).date()),
        "fim_validacao": str(pd.Timestamp(corte_val).date()),
    }
    return partes, cortes


def ajustar_normalizacao(X_treino):
    log_x = np.log1p(X_treino)
    media = float(log_x.mean())
    desvio = float(log_x.std())
    return media, max(desvio, 1e-8)


def normalizar_X(X, media, desvio):
    return ((np.log1p(X) - media) / desvio).astype(np.float32)

## Modelo GRU com conservação do total diário

In [4]:
class GRUDesagregadora(nn.Module):
    def __init__(self, hidden_size, n_outputs, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_outputs = n_outputs
        self.num_layers = num_layers
        self.gru = nn.GRU(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0 if num_layers == 1 else 0.1,
        )
        self.fc = nn.Linear(hidden_size, n_outputs)

    def forward(self, x, total_central):
        sequencia, h_n = self.gru(x)
        logits = self.fc(sequencia[:, -1, :])
        fracoes = torch.softmax(logits, dim=1)
        previsao = fracoes * total_central[:, None]
        return previsao, fracoes, h_n


def criar_loader(parte, media, desvio, batch_size, shuffle):
    x = torch.from_numpy(normalizar_X(parte["X"], media, desvio))
    y = torch.from_numpy(parte["Y"].astype(np.float32))
    total = y.sum(dim=1)
    dataset = TensorDataset(x, y, total)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


@torch.no_grad()
def perda_loader(modelo, loader, criterio, device):
    modelo.eval()
    perdas = []
    for x, y, total in loader:
        x, y, total = x.to(device), y.to(device), total.to(device)
        pred, _, _ = modelo(x, total)
        perdas.append(criterio(pred, y).item() * len(x))
    return sum(perdas) / len(loader.dataset)


@torch.no_grad()
def prever(modelo, parte, media, desvio, device, batch_size=256):
    loader = criar_loader(parte, media, desvio, batch_size, shuffle=False)
    modelo.eval()
    observados, previstos, fracoes = [], [], []
    for x, y, total in loader:
        pred, frac, _ = modelo(x.to(device), total.to(device))
        observados.append(y.numpy())
        previstos.append(pred.cpu().numpy())
        fracoes.append(frac.cpu().numpy())
    return np.vstack(observados), np.vstack(previstos), np.vstack(fracoes)

## Métricas, referências e treinamento

In [5]:
def calcular_metricas(obs, pred, intervalo_horas):
    o, p = obs.ravel(), pred.ravel()
    rmse = float(np.sqrt(mean_squared_error(o, p)))
    mae = float(mean_absolute_error(o, p))
    correlacao = float(np.corrcoef(o, p)[0, 1]) if np.std(o) > 0 and np.std(p) > 0 else np.nan
    pico_obs = obs.max(axis=1)
    pico_pred = pred.max(axis=1)
    pico_mae = float(mean_absolute_error(pico_obs, pico_pred))
    pico_rel = float(np.mean(np.abs(pico_pred - pico_obs) / np.maximum(pico_obs, 1e-6)) * 100)
    atraso = np.abs(np.argmax(pred, axis=1) - np.argmax(obs, axis=1)) * intervalo_horas
    conservacao = np.abs(pred.sum(axis=1) - obs.sum(axis=1))
    return {
        "MAE_mm": mae,
        "RMSE_mm": rmse,
        "correlacao": correlacao,
        "MAE_pico_mm": pico_mae,
        "erro_relativo_pico_pct": pico_rel,
        "erro_temporal_pico_h_mediana": float(np.median(atraso)),
        "erro_conservacao_mm_max": float(conservacao.max()),
    }


def previsao_climatologica(parte_treino, parte_teste):
    y_train = parte_treino["Y"]
    totais_train = y_train.sum(axis=1, keepdims=True)
    frac_train = y_train / np.maximum(totais_train, 1e-8)
    meses_train = pd.DatetimeIndex(parte_treino["datas"]).month
    media_geral = frac_train.mean(axis=0)
    perfis = {
        mes: frac_train[meses_train == mes].mean(axis=0)
        for mes in np.unique(meses_train)
    }
    meses_teste = pd.DatetimeIndex(parte_teste["datas"]).month
    frac = np.vstack([perfis.get(m, media_geral) for m in meses_teste])
    return frac * parte_teste["Y"].sum(axis=1, keepdims=True)


def treinar_configuracao(
    intervalo_horas=1,
    janela_dias=5,
    hidden_size=32,
    num_layers=1,
    epocas=200,
    batch_size=64,
    lr=1e-3,
    paciencia=20,
    repeticao=1,
    mostrar_grafico=True,
):
    if intervalo_horas not in {1, 2, 4}:
        raise ValueError("Escolha intervalo de 1, 2 ou 4 horas.")
    if janela_dias not in {3, 5, 7}:
        raise ValueError("Escolha janela de 3, 5 ou 7 dias.")

    semente = SEMENTE_BASE + int(repeticao)
    fixar_semente(semente)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_outputs = 24 // intervalo_horas
    print(f"Dispositivo: {device} | janela={janela_dias} d | saídas={n_outputs} ({intervalo_horas} h)")

    horario = carregar_horario(CAMINHO_DADOS)
    amostras = construir_amostras(horario, janela_dias, intervalo_horas, LIMIAR_CHUVA_MM)
    partes, cortes = dividir_cronologicamente(amostras)
    media, desvio = ajustar_normalizacao(partes["treino"]["X"])

    print("Amostras:", {k: len(v["X"]) for k, v in partes.items()})
    print("Cortes:", cortes)

    loaders = {
        "treino": criar_loader(partes["treino"], media, desvio, batch_size, True),
        "validacao": criar_loader(partes["validacao"], media, desvio, batch_size, False),
    }
    modelo = GRUDesagregadora(hidden_size, n_outputs, num_layers).to(device)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    criterio = nn.MSELoss()

    melhor_estado = None
    melhor_val = math.inf
    sem_melhora = 0
    historico = []
    inicio = time.time()

    for epoca in range(1, epocas + 1):
        modelo.train()
        soma = 0.0
        for x, y, total in loaders["treino"]:
            x, y, total = x.to(device), y.to(device), total.to(device)
            otimizador.zero_grad()
            pred, _, _ = modelo(x, total)
            perda = criterio(pred, y)
            perda.backward()  # BPTT: o gradiente atravessa os dias da janela
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
            otimizador.step()
            soma += perda.item() * len(x)

        perda_treino = soma / len(loaders["treino"].dataset)
        perda_val = perda_loader(modelo, loaders["validacao"], criterio, device)
        historico.append({"epoca": epoca, "MSE_treino": perda_treino, "MSE_validacao": perda_val})

        if perda_val < melhor_val - 1e-8:
            melhor_val = perda_val
            melhor_estado = copy.deepcopy(modelo.state_dict())
            sem_melhora = 0
        else:
            sem_melhora += 1

        if epoca == 1 or epoca % 10 == 0:
            print(f"Época {epoca:03d} | treino={perda_treino:.5f} | validação={perda_val:.5f}")
        if sem_melhora >= paciencia:
            print(f"Parada antecipada na época {epoca}.")
            break

    modelo.load_state_dict(melhor_estado)
    obs, pred, frac = prever(modelo, partes["teste"], media, desvio, device)
    uniforme = np.ones_like(obs) * (obs.sum(axis=1, keepdims=True) / n_outputs)
    climatologica = previsao_climatologica(partes["treino"], partes["teste"])

    linhas_metricas = []
    for metodo, valores in [("GRU", pred), ("uniforme", uniforme), ("climatologico_mensal", climatologica)]:
        linha = {"metodo": metodo, **calcular_metricas(obs, valores, intervalo_horas)}
        linhas_metricas.append(linha)
    metricas = pd.DataFrame(linhas_metricas)
    display(metricas)

    nome = f"gru_{intervalo_horas}h_{janela_dias}d_{hidden_size}oc_r{repeticao}"
    PASTA_MODELOS.mkdir(parents=True, exist_ok=True)
    PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
    caminho_modelo = PASTA_MODELOS / f"{nome}.pt"

    config = {
        "nome": nome,
        "intervalo_horas": intervalo_horas,
        "n_outputs": n_outputs,
        "janela_dias": janela_dias,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "media_log_treino": media,
        "desvio_log_treino": desvio,
        "limiar_chuva_mm": LIMIAR_CHUVA_MM,
        "fim_treino": cortes["fim_treino"],
        "fim_validacao": cortes["fim_validacao"],
        "semente": semente,
        "dados": str(CAMINHO_DADOS),
        "melhor_MSE_validacao": float(melhor_val),
    }
    torch.save({"state_dict": modelo.state_dict(), "config": config}, caminho_modelo)

    hist_df = pd.DataFrame(historico)
    hist_df.to_csv(PASTA_RESULTADOS / f"{nome}_historico.csv", index=False)
    metricas.to_csv(PASTA_RESULTADOS / f"{nome}_metricas.csv", index=False)

    prev = pd.DataFrame({"data": pd.to_datetime(partes["teste"]["datas"])})
    for i in range(n_outputs):
        prev[f"obs_{i+1:02d}"] = obs[:, i]
        prev[f"pred_{i+1:02d}"] = pred[:, i]
    prev.to_csv(PASTA_RESULTADOS / f"{nome}_previsoes_teste.csv", index=False)

    if mostrar_grafico:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
        axes[0].plot(hist_df["epoca"], hist_df["MSE_treino"], label="treino")
        axes[0].plot(hist_df["epoca"], hist_df["MSE_validacao"], label="validação")
        axes[0].set(xlabel="Época", ylabel="MSE", title="Curva de aprendizagem")
        axes[0].legend()

        totais_teste = obs.sum(axis=1)
        idx = int(np.argmax(totais_teste))
        x_blocos = np.arange(n_outputs) * intervalo_horas
        axes[1].bar(x_blocos - 0.18 * intervalo_horas, obs[idx], width=0.36 * intervalo_horas, label="observado")
        axes[1].bar(x_blocos + 0.18 * intervalo_horas, pred[idx], width=0.36 * intervalo_horas, label="GRU")
        axes[1].set(
            xlabel="Início do intervalo (h)", ylabel=f"mm/{intervalo_horas}h",
            title=f"Maior total do teste: {pd.Timestamp(partes['teste']['datas'][idx]).date()}"
        )
        axes[1].legend()
        plt.show()

    print(f"Modelo salvo: {caminho_modelo.resolve()}")
    print(f"Tempo: {time.time() - inicio:.1f} s")
    return {"modelo": modelo, "config": config, "metricas": metricas, "partes": partes, "caminho": caminho_modelo}

## Executar uma configuração pela célula

In [6]:
# Descomente para treinar usando os valores da configuração inicial:
# resultado = treinar_configuracao(
#     intervalo_horas=INTERVALO_HORAS,
#     janela_dias=JANELA_DIAS,
#     hidden_size=NEURONIOS_OCULTOS,
#     num_layers=NUM_CAMADAS,
#     epocas=EPOCAS,
#     batch_size=BATCH_SIZE,
#     lr=TAXA_APRENDIZAGEM,
#     paciencia=PACIENCIA,
#     repeticao=REPETICAO,
# )

## Interface interativa

In [7]:
try:
    import ipywidgets as widgets

    w_resolucao = widgets.Dropdown(
        options=[("24 saídas — 1 hora", 1), ("12 saídas — 2 horas", 2), ("6 saídas — 4 horas", 4)],
        value=1,
        description="Saída:",
        style={"description_width": "90px"},
    )
    w_janela = widgets.ToggleButtons(options=[3, 5, 7], value=5, description="Dias:")
    w_hidden = widgets.Dropdown(options=[8, 16, 32, 64, 128], value=32, description="Ocultos:")
    w_camadas = widgets.Dropdown(options=[1, 2], value=1, description="Camadas:")
    w_epocas = widgets.IntSlider(value=200, min=20, max=500, step=20, description="Épocas:")
    w_repeticao = widgets.IntSlider(value=1, min=1, max=5, step=1, description="Repetição:")
    botao_um = widgets.Button(description="Treinar configuração", button_style="primary", icon="play")
    botao_grade = widgets.Button(description="Treinar grade 3 × 3", button_style="warning", icon="table")
    saida = widgets.Output()

    def ao_treinar_um(_):
        with saida:
            clear_output(wait=True)
            treinar_configuracao(
                intervalo_horas=w_resolucao.value,
                janela_dias=w_janela.value,
                hidden_size=w_hidden.value,
                num_layers=w_camadas.value,
                epocas=w_epocas.value,
                batch_size=BATCH_SIZE,
                lr=TAXA_APRENDIZAGEM,
                paciencia=PACIENCIA,
                repeticao=w_repeticao.value,
            )

    def ao_treinar_grade(_):
        with saida:
            clear_output(wait=True)
            resumos = []
            total = 9
            contador = 0
            for intervalo in [1, 2, 4]:
                for janela in [3, 5, 7]:
                    contador += 1
                    print(f"\n===== Modelo {contador}/{total}: {intervalo} h, {janela} dias =====")
                    r = treinar_configuracao(
                        intervalo_horas=intervalo,
                        janela_dias=janela,
                        hidden_size=w_hidden.value,
                        num_layers=w_camadas.value,
                        epocas=w_epocas.value,
                        batch_size=BATCH_SIZE,
                        lr=TAXA_APRENDIZAGEM,
                        paciencia=PACIENCIA,
                        repeticao=w_repeticao.value,
                        mostrar_grafico=False,
                    )
                    linha = r["metricas"].query("metodo == 'GRU'").iloc[0].to_dict()
                    linha.update({"intervalo_horas": intervalo, "janela_dias": janela})
                    resumos.append(linha)
            tabela = pd.DataFrame(resumos)
            display(tabela.sort_values("RMSE_mm"))
            PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
            tabela.to_csv(PASTA_RESULTADOS / "resumo_grade_3x3.csv", index=False)

    botao_um.on_click(ao_treinar_um)
    botao_grade.on_click(ao_treinar_grade)
    display(widgets.VBox([
        widgets.HTML("<b>Configuração da GRU</b>"),
        widgets.HBox([w_resolucao, w_janela]),
        widgets.HBox([w_hidden, w_camadas]),
        widgets.HBox([w_epocas, w_repeticao]),
        widgets.HBox([botao_um, botao_grade]),
        saida,
    ]))
except ImportError:
    print("ipywidgets não está instalado. Use a célula anterior ou instale ipywidgets.")

## O que será usado no notebook 3

Cada arquivo `.pt` contém:

- pesos da GRU e da camada de saída;
- número de saídas, janela, unidades ocultas e camadas;
- média e desvio usados na normalização;
- datas dos cortes cronológicos;
- limiar de dia chuvoso e semente.

Não altere manualmente o arquivo do modelo. O notebook 3 lê essas informações para reconstruir exatamente as amostras de teste e verificar numericamente as equações das portas.